In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install "transformers>=4.48,<4.56" evaluate rouge_score bert_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 166.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24984 sha256=32219683469076f2c8f99362a5a8c925b7845927587f7f1a8d221b49e2da135e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.2

In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 110.5 MB/s eta 0:00:00


# Load files from Training

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re

out_dir  = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run"
work_dir = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_work"
csv_dir  = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/Prot2TextDataset"
baseline_predict_path  = "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/Full_run/test_baseline_predictions.pkl"
baseline_metric_path = ("/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/multitask/p2t_aux_checkpoints/generations/baseline_vs_finetuned_n4203.json")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)
print("paths configured")

paths configured


In [ ]:
#generation JSONs from the training notebook
def load_gen(fname):
    path = os.path.join(out_dir, fname)
    with open(path) as f:
        data = json.load(f)
    print(f"  {len(data):,} proteins <- {fname}")
    return data

print("loading generation files...")
gen = {
    "seq_only" : load_gen("generation_seq_only.json"),
    "without"  : load_gen("generation_test_noretrieval.json"),
    "with"     : load_gen("generation_test_retrieval.json"),
}

print("\nloading external pkl baseline...")
with open(baseline_predict_path, "rb") as f:
    baseline_predict = pickle.load(f)
print(f"  {len(baseline_predict):,} accessions in external pkl baseline")

print("\nloading external metrics JSON...")
with open(baseline_metric_path) as f:
    baseline_metric = json.load(f)
ext_matched  = set(baseline_metric["matched_names"])
baseline_metrics  = baseline_metric["baseline"]
print(f"  {len(ext_matched):,} accessions in external JSON")
print(f"  external baseline exact_match : {baseline_metrics['exact_match']:.4f}")

print("\nloading supporting files...")
with open(os.path.join(work_dir, "prompt_hits.json")) as f:
    prompt_hits = json.load(f)
print(f"  prompt_hits: {len(prompt_hits['test']):,} test entries")

test_csv = pd.read_csv(os.path.join(csv_dir, "test.csv"))
print(f"  test.csv   : {len(test_csv):,} rows")

with open(os.path.join(out_dir, "metrics.json")) as f:
    agg_metrics = json.load(f)

seq_only_path = os.path.join(out_dir, "metrics_seq_only.json")
agg_metrics_seq = json.load(open(seq_only_path)) if os.path.exists(seq_only_path) else {}

strat = pd.read_csv(os.path.join(out_dir, "stratified_metrics.csv"))
print(f"  stratified : {len(strat)} bins")

history_path = os.path.join(out_dir, "history.json")
history = json.load(open(history_path)) if os.path.exists(history_path) else []

loading generation files...
  4,203 proteins <- generation_seq_only.json
  4,203 proteins <- generation_test_noretrieval.json
  4,203 proteins <- generation_test_retrieval.json

loading external pkl baseline...
  4,203 accessions in external pkl baseline

loading external metrics JSON...
  4,203 accessions in external JSON
  external baseline exact_match : 0.3990

loading supporting files...
  prompt_hits: 4,163 test entries
  test.csv   : 4,203 rows
  stratified : 5 bins


# Build master per-protein DataFrame

In [ ]:
# Use accessions present in all non-empty local generation conditions as index.
local_conditions = {k: v for k, v in gen.items() if v}
all_accs = set.intersection(*[set(v.keys()) for v in local_conditions.values()])
print(f"{len(all_accs):,} accessions in all local conditions")
print(f"{len(ext_matched):,} accessions in external JSON")
print(f"{len(set(baseline_predict.keys()) & all_accs):,} baseline_predict accessions overlap with local")

test_meta = test_csv.set_index("AlphaFoldDB")
cond_order = [c for c in ["baseline", "seq_only", "without", "with"] if c in local_conditions]

rows = []
for acc in all_accs:
    hits = prompt_hits["test"].get(acc, [])
    best = hits[0] if hits else None

    row = {
        "accession"       : acc,
        "taxon"           : test_meta.loc[acc, "taxon"] if acc in test_meta.index else "",
        "reference"       : local_conditions["with"][acc]["true"],
        "best_hit_target" : best["target"]  if best else None,
        "best_hit_fident" : best["fident"]  if best else None,
        "best_hit_qcov"   : best["qcov"]    if best else None,
        "best_hit_evalue" : best["evalue"]  if best else None,
        "best_hit_desc"   : best["desc"]    if best else None,
        "best_hit_org"    : best.get("org") if best else None,
    }

    # local predictions
    for cond in cond_order:
        row[f"pred_{cond}"] = local_conditions[cond][acc]["pred"]

    # baseline prediction (plain string, no reference stored here)
    row["pred_baseline"] = baseline_predict.get(acc)

    rows.append(row)

df = pd.DataFrame(rows).set_index("accession")

def fident_bin(f):
    if f is None or (isinstance(f, float) and np.isnan(f)): return "no hit"
    if f < 0.20: return "<20%"
    if f < 0.30: return "20-30%"
    if f < 0.50: return "30-50%"
    return ">50%"

df["fident_bin"] = df["best_hit_fident"].apply(fident_bin)
print(f"DataFrame shape: {df.shape}")
df.head(3)

4,203 accessions in all local conditions
4,203 accessions in external JSON
4,203 baseline_predict accessions overlap with local
DataFrame shape: (4203, 13)


,taxon,reference,best_hit_target,best_hit_fident,best_hit_qcov,best_hit_evalue,best_hit_desc,best_hit_org,pred_seq_only,pred_without,pred_with,pred_baseline,fident_bin
accession,,,,,,,,,,,,,
Q04W49,Leptospira,Necessary for normal cell division and for the maintenance of normal septation.,B0SC16,0.4690,0.8870,0.0000,Necessary for normal cell division and for the maintenance of normal septation.,None,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,30-50%
Q8TR92,Methanosarcina,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Q8PUP4,0.9060,0.9850,0.0000,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,None,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,>50%
O95905,Homo,Regulator of p53/TP53 stability and function. Inhibits MDM2-mediated degradation of p53/TP53 possibly by cooperating...,Q9W032,0.3050,0.9920,0.0000,Required in both the follicle cells and the germline for oocyte development.,None,May be required for efficient export of polyadenylated RNA.,May be required for efficient export of polyadenylated RNA.,May be required for efficient export of polyadenylated RNA.,Required in both the follicle cells and the germline for oocyte development.,30-50%


# Aggregate metrics comparison

In [ ]:
# Helper: pull the standard scalar metrics from any metrics dict that matches
# the structure produced by compute_metrics() in the training notebook.
def extract(m):
    if not m: return {}
    return {
        "exact_match" : m.get("exact_match", float("nan")),
        "bleu2"       : m.get("bleu2", {}).get("bleu", float("nan")),
        "bleu4"       : m.get("bleu4", {}).get("bleu", float("nan")),
        "rouge1"      : m.get("rouge", {}).get("rouge1", float("nan")),
        "rougeL"      : m.get("rouge", {}).get("rougeL", float("nan")),
        "bert_rob_f1" : m.get("bert", {}).get("roberta-large", {}).get("f1", float("nan")),
        "bert_bio_f1" : m.get("bert", {}).get("biobert-large", {}).get("f1", float("nan")),
    }

summary = pd.DataFrame({
    "baseline"  : extract(baseline_metrics),
    "seq_only"      : extract(agg_metrics_seq),
    "without"       : extract(agg_metrics.get("without_retrieval", {})),
    "with"          : extract(agg_metrics.get("with_retrieval", {})),
}).T

summary.index.name = "condition"
print("Aggregate metrics across all conditions:")
print(summary.to_string())
summary

Aggregate metrics across all conditions:
           exact_match  bleu2  bleu4  rouge1  rougeL  bert_rob_f1  bert_bio_f1
condition                                                                     
baseline        0.3990 0.4458 0.4056  0.5785  0.5588       0.9185       0.8665
seq_only        0.3583 0.3712 0.3355  0.5303  0.5113       0.9109       0.8518
without         0.3605 0.3757 0.3405  0.5327  0.5143       0.9113       0.8530
with            0.3685 0.3829 0.3478  0.5420  0.5234       0.9129       0.8551


,exact_match,bleu2,bleu4,rouge1,rougeL,bert_rob_f1,bert_bio_f1
condition,,,,,,,
baseline,0.3990,0.4458,0.4056,0.5785,0.5588,0.9185,0.8665
seq_only,0.3583,0.3712,0.3355,0.5303,0.5113,0.9109,0.8518
without,0.3605,0.3757,0.3405,0.5327,0.5143,0.9113,0.8530
with,0.3685,0.3829,0.3478,0.5420,0.5234,0.9129,0.8551


Stratified metrics by Foldseek identity bin

In [ ]:
# The stratified CSV was produced by the training notebook and covers only
# the with/without conditions.  Display with coloured deltas.
delta_cols = [c for c in strat.columns if c.startswith("d_")]

def colour_delta(v):
    if pd.isna(v): return ""
    return "color: green" if v > 0 else ("color: red" if v < 0 else "")

strat.style.applymap(colour_delta, subset=delta_cols).format(
    {c: "{:.4f}" for c in strat.select_dtypes(float).columns}
)

/tmp/ipykernel_3384/2567408585.py:9: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  strat.style.applymap(colour_delta, subset=delta_cols).format(


,bin,n,bleu4_with,rougeL_with,bert_rob_with,bert_bio_with,bleu4_without,rougeL_without,bert_rob_without,bert_bio_without,d_bleu4,d_rougeL,d_bert_rob,d_bert_bio
0,no hit,40,0.3960,0.4467,0.8978,0.8264,0.3965,0.4472,0.8993,0.8272,-0.0005,-0.0005,-0.0015,-0.0008
1,<20%,638,0.1459,0.2562,0.8644,0.7699,0.1354,0.2510,0.8633,0.7694,0.0105,0.0052,0.0011,0.0005
2,20-30%,886,0.2371,0.4024,0.8909,0.8162,0.2398,0.3948,0.8900,0.8158,-0.0027,0.0075,0.0009,0.0003
3,30-50%,1457,0.3438,0.5532,0.9179,0.8658,0.3336,0.5423,0.9160,0.8630,0.0103,0.0109,0.0019,0.0029
4,>50%,1182,0.5548,0.7257,0.9499,0.9179,0.5448,0.7150,0.9479,0.9146,0.0100,0.0106,0.0019,0.0033


Per-protein prediction DataFrame

In [ ]:
# Full table with reference, all predictions, and hit metadata.
text_cols = ["reference", "best_hit_desc", "pred_baseline"] +             [f"pred_{c}" for c in cond_order]
meta_cols = ["taxon", "fident_bin", "best_hit_fident", "best_hit_qcov",
             "best_hit_evalue", "best_hit_target", "best_hit_org"]

df_display = df[meta_cols + text_cols].copy()
df_display.head(10)

,taxon,fident_bin,best_hit_fident,best_hit_qcov,best_hit_evalue,best_hit_target,best_hit_org,reference,best_hit_desc,pred_baseline,pred_seq_only,pred_without,pred_with
accession,,,,,,,,,,,,,
Q04W49,Leptospira,30-50%,0.4690,0.8870,0.0000,B0SC16,None,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.
Q8TR92,Methanosarcina,>50%,0.9060,0.9850,0.0000,Q8PUP4,None,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...
O95905,Homo,30-50%,0.3050,0.9920,0.0000,Q9W032,None,Regulator of p53/TP53 stability and function. Inhibits MDM2-mediated degradation of p53/TP53 possibly by cooperating...,Required in both the follicle cells and the germline for oocyte development.,Required in both the follicle cells and the germline for oocyte development.,May be required for efficient export of polyadenylated RNA.,May be required for efficient export of polyadenylated RNA.,May be required for efficient export of polyadenylated RNA.
Q70KD0,Micromonospora,>50%,0.5500,0.9520,0.0000,Q2MF16,None,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).
P48013,Schizosaccharomyces,>50%,0.6480,0.9930,0.0000,P26306,None,"Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ..."
P22855,Saccharomyces,30-50%,0.4750,1.0000,0.0000,Q9UT61,None,Degrades free oligosaccharides in the vacuole.,Degrades free oligosaccharides in the vacuole.,Degrades free oligosaccharides in the vacuole.,Degrades free oligosaccharides in the vacuole.,Degrades free oligosaccharides in the vacuole.,Degrades free oligosaccharides in the vacuole.
Q06597,Saccharomyces,20-30%,0.2650,0.9690,0.0000,Q8GY31,None,Involved in resistance to arsenic compounds.,Arsenate reductase 

In [ ]:
# Filter to a specific identity bin.
# Options: "no hit", "<20%", "20-30%", "30-50%", ">50%"
SHOW_BIN = ">50%"

df_bin = df_display[df_display["fident_bin"] == SHOW_BIN].copy()
print(f"{len(df_bin)} proteins in the {SHOW_BIN!r} bin")
df_bin.head(10)

1182 proteins in the '>50%' bin


,taxon,fident_bin,best_hit_fident,best_hit_qcov,best_hit_evalue,best_hit_target,best_hit_org,reference,best_hit_desc,pred_baseline,pred_seq_only,pred_without,pred_with
accession,,,,,,,,,,,,,
Q8TR92,Methanosarcina,>50%,0.9060,0.9850,0.0000,Q8PUP4,None,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...
Q70KD0,Micromonospora,>50%,0.5500,0.9520,0.0000,Q2MF16,None,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).
P48013,Schizosaccharomyces,>50%,0.6480,0.9930,0.0000,P26306,None,"Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ..."
Q5W7C1,Oryza sativa,>50%,0.7100,0.9370,0.0000,Q9ZUT3,None,Associates with STAR2 to form a functional transmembrane ABC transporter required for detoxification of aluminum (Al...,"Required for aluminum (Al) resistance/tolerance, probably by translocating Al from sensitive tissues such as growing...","Required for aluminum (Al) resistance/tolerance, probably by translocating Al from sensitive tissues such as growing...","Required for aluminum (Al) resistance/tolerance, probably by translocating Al from sensitive tissues such as growing...","Required for aluminum (Al) resistance/tolerance, probably by translocating Al from sensitive tissues such as growing...","Required for aluminum (Al) resistance/tolerance, probably by translocating Al from sensitive tissues such as growing..."
Q92PW7,Sinorhizobium,>50%,0.6110,0.9530,0.0000,Q11HZ9,None,Phosphorylation of dTMP to form dTDP in both de novo and salvage pathways of dTTP synthesis.,Phosphorylation of dTMP to form dTDP in both de novo and salvage pathways of dTTP synthesis.,Phosphorylation of dTMP to form dTDP in both de novo and salvage pathways of dTTP synthesis.,Phosphorylation of dTMP to form dTDP in both de novo and salvage pathways of dTTP synthesis.,Phosphorylation of dTMP to form dTDP in both de novo and salvage pathways of dTTP synthesis.,Phosphorylation of dTMP to form dTDP in both de novo and salvage pathways of dTTP synthesis.
B8MPW2,Talaromyces sect. Talaromyces,>50%,0.7270,0.9570,0.0000,B6Q2V1,None,Probable mitochondrial mRNA stab

In [ ]:
N_SHOW = 5
FILTER = df["fident_bin"] == ">50%"    # change as needed

all_pred_conds = ["baseline"] + cond_order
sample = df[FILTER].sample(min(N_SHOW, FILTER.sum()), random_state=42)

for acc, row in sample.iterrows():
    print("=" * 80)
    print(f"ACCESSION : {acc}")
    print(f"TAXON     : {row['taxon']}")
    fid = row['best_hit_fident']
    if fid is not None and not (isinstance(fid, float) and np.isnan(fid)):
        print(f"HIT       : {row['best_hit_target']}  {fid*100:.0f}% id  org={row['best_hit_org']}")
        print(f"HIT DESC  : {str(row['best_hit_desc'])[:200]}")
    print()
    print(f"REFERENCE : {row['reference']}")
    print()
    for cond in all_pred_conds:
        col = f"pred_{cond}"
        if col in row.index and row[col] is not None and not (isinstance(row[col], float) and np.isnan(row[col])):
            print(f"PRED ({cond:<14}): {row[col]}")
    print()

ACCESSION : P08709
TAXON     : Homo
HIT       : P22457  67% id  org=Bos
HIT DESC  : Initiates the extrinsic pathway of blood coagulation. Serine protease that circulates in the blood in a zymogen form. Factor VII is converted to factor VIIa by factor Xa, factor XIIa, factor IXa, or t

REFERENCE : Initiates the extrinsic pathway of blood coagulation. Serine protease that circulates in the blood in a zymogen form. Factor VII is converted to factor VIIa by factor Xa, factor XIIa, factor IXa, or thrombin by minor proteolysis. In the presence of tissue factor and calcium ions, factor VIIa then converts factor X to factor Xa by limited proteolysis. Factor VIIa will also convert factor IX to factor IXa in the presence of tissue factor and calcium.

PRED (baseline      ): Protein C is a vitamin K-dependent serine protease that regulates blood coagulation by inactivating factors Va and VIIIa in the presence of calcium ions and phospholipids. Exerts a protective effect on the endothelial cell ba

# Export results table

In [ ]:
df_copy = pd.DataFrame(copy_rows).set_index("accession")
copy_summary = df_copy.groupby("fident_bin").mean(numeric_only=True).round(3)
df_export = df.merge(df_copy.drop(columns="fident_bin", errors="ignore"),
                     left_index=True, right_index=True, how="left")

export_path = os.path.join(out_dir, "all_predictions_analysis.csv")
df_export.to_csv(export_path)
print(f"saved {len(df_export):,} rows → {export_path}")
print(f"columns: {list(df_export.columns)}")

saved 4,203 rows → /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run/all_predictions_analysis.csv
columns: ['taxon', 'reference', 'best_hit_target', 'best_hit_fident', 'best_hit_qcov', 'best_hit_evalue', 'best_hit_desc', 'best_hit_org', 'pred_seq_only', 'pred_without', 'pred_with', 'pred_baseline', 'fident_bin', 'jaccard_with_without', 'cross_taxon', 'reference_overlap', 'seq_only_overlap', 'without_overlap', 'with_overlap', 'baseline_overlap']


# Recompute Metrics on Additional Data and on Individual Protein Level

In [ ]:
# Max tokens fed to the BERTscore model
bert_truncate_length = 495
roberta_model        = "FacebookAI/roberta-large"
biobert_model        = "dmis-lab/biobert-large-cased-v1.1"
biobert_num_layers   = 24

import evaluate
from transformers import BertTokenizer, RobertaTokenizer


def compute_exact_match(predictions, references):
    # Fraction of predictions that exactly match their reference after
    # lowercasing and stripping all non-word characters. A very strict metric;
    # non-zero scores indicate verbatim agreement.
    def normalize(t):
        return re.sub(r"[^\w]", "", t.lower())
    return sum(normalize(p) == normalize(r) for p, r in zip(predictions, references)) / len(predictions)


def compute_bleu(predictions, references, max_order=4):
    # BLEU score measuring n-gram precision up to max_order
    return evaluate.load("bleu").compute(
        predictions=predictions, references=references, max_order=max_order)


def compute_rouge(predictions, references):
    # ROUGE scores measuring recall-oriented n-gram and subsequence overlap.
    # Returns rouge1 (unigram), rouge2 (bigram), and rougeL (LCS-based).
    return evaluate.load("rouge").compute(predictions=predictions, references=references)


def _truncate(texts, tokenizer, max_length=bert_truncate_length):
    # Tokenise and decode texts to fit within the BERTScore model's limit.
    ids = tokenizer(texts, padding="max_length", truncation=True,
                    max_length=max_length, return_tensors="pt")["input_ids"]
    return tokenizer.batch_decode(ids, skip_special_tokens=True)


def compute_bert_score(predictions, references):
    # BERTScore using both RoBERTa-large and BioBERT-large. BERTScore computes
    # token-level cosine similarities between contextual embeddings of prediction
    # and reference tokens, aggregated into precision, recall, and F1.
    bert, results = evaluate.load("bertscore"), {}

    rt = RobertaTokenizer.from_pretrained(roberta_model)
    r = bert.compute(predictions=_truncate(predictions, rt),
                     references=_truncate(references, rt), lang="en")
    results["roberta-large"] = {k: sum(r[k]) / len(r[k]) for k in ("precision", "recall", "f1")}

    bt = BertTokenizer.from_pretrained(biobert_model)
    b = bert.compute(predictions=_truncate(predictions, bt),
                     references=_truncate(references, bt),
                     model_type=biobert_model, num_layers=biobert_num_layers)
    results["biobert-large"] = {k: sum(b[k]) / len(b[k]) for k in ("precision", "recall", "f1")}
    return results


def compute_metrics(predictions, references, bert_score=True, verbose=True):
    # Compute the full metric suite: exact match, BLEU-2, BLEU-4, ROUGE, and
    # BERTScore. bert_score=False skips the expensive BERTScore computation
    # (used for stratified bin analysis where it would be called many times).
    out = {"n": len(predictions)}
    out["exact_match"] = compute_exact_match(predictions, references)
    out["bleu2"] = compute_bleu(predictions, references, max_order=2)
    out["bleu4"] = compute_bleu(predictions, references, max_order=4)
    out["rouge"] = compute_rouge(predictions, references)
    if bert_score:
        out["bert"] = compute_bert_score(predictions, references)
    if verbose:
        print(f"n            : {out['n']}")
        print(f"exact match  : {out['exact_match']:.4f}")
        print(f"BLEU-2       : {out['bleu2']['bleu']:.4f}")
        print(f"BLEU-4       : {out['bleu4']['bleu']:.4f}")
        print(f"ROUGE-1/2/L  : {out['rouge']['rouge1']:.4f} / "
              f"{out['rouge']['rouge2']:.4f} / {out['rouge']['rougeL']:.4f}")
        if bert_score:
            for m, v in out["bert"].items():
                print(f"BERTScore {m:14s} P/R/F1: {v['precision']:.4f} / "
                      f"{v['recall']:.4f} / {v['f1']:.4f}")
    return out


def score_results(results, **kw):
    # Helper that calls compute_metrics on a generation result dict.
    accs = list(results.keys())
    return accs, compute_metrics([results[a]["pred"] for a in accs],
                                 [results[a]["true"] for a in accs], **kw)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
# ── Cell 1: stratified by identity bin, baseline vs with ─────────────────────
# Identity bins for stratified analysis. Test proteins are binned by best-hit sequence identity (fident).
ident_bins   = [(0.00, 0.20, "<20%"), (0.20, 0.30, "20-30%"),
                (0.30, 0.50, "30-50%"), (0.50, 1.01, ">50%")]
# Bins with fewer proteins than this are reported as count only
min_bin_size = 20
def best_fident(acc):
    # Return best-hit sequence identity for a test accession, or None.
    hs = prompt_hits["test"].get(acc, [])
    return hs[0]["fident"] if hs else None


def bin_of(acc):
    # Map a test accession to its identity bin label.
    f = best_fident(acc)
    if f is None:
        return "no hit"
    for lo, hi, name in ident_bins:
        if lo <= f < hi:
            return name
    return ident_bins[-1][2]

rows = []
for name in ["no hit"] + [b[2] for b in ident_bins]:
    accs = [a for a in df.index if bin_of(a) == name and a in baseline_predict]
    if len(accs) < min_bin_size:
        rows.append({"bin": name, "n": len(accs)})
        continue
    r = {"bin": name, "n": len(accs)}
    refs = [df.loc[a, "reference"] for a in accs]
    for tag, preds in [
        ("with",     [df.loc[a, "pred_with"] for a in accs]),
        ("baseline", [baseline_predict[a]     for a in accs]),
    ]:
        m = compute_metrics(preds, refs, bert_score=True, verbose=False)
        r[f"bleu4_{tag}"]    = m["bleu4"]["bleu"]
        r[f"rougeL_{tag}"]   = m["rouge"]["rougeL"]
        r[f"bert_rob_{tag}"] = m["bert"]["roberta-large"]["f1"]
        r[f"bert_bio_{tag}"] = m["bert"]["biobert-large"]["f1"]
    r["d_bleu4"]    = r["bleu4_with"]    - r["bleu4_baseline"]
    r["d_rougeL"]   = r["rougeL_with"]   - r["rougeL_baseline"]
    r["d_bert_rob"] = r["bert_rob_with"]  - r["bert_rob_baseline"]
    r["d_bert_bio"] = r["bert_bio_with"]  - r["bert_bio_baseline"]
    rows.append(r)

strat_baseline_vs_with = pd.DataFrame(rows)
strat_baseline_vs_with.to_csv(os.path.join(out_dir, "stratified_baseline_vs_with.csv"), index=False)
strat_baseline_vs_with

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

,bin,n,bleu4_with,rougeL_with,bert_rob_with,bert_bio_with,bleu4_baseline,rougeL_baseline,bert_rob_baseline,bert_bio_baseline,d_bleu4,d_rougeL,d_bert_rob,d_bert_bio
0,no hit,40,0.3960,0.4425,0.8978,0.8264,0.4292,0.4748,0.9047,0.8333,-0.0332,-0.0323,-0.0068,-0.0070
1,<20%,638,0.1459,0.2564,0.8644,0.7699,0.1673,0.2741,0.8655,0.7743,-0.0214,-0.0177,-0.0010,-0.0043
2,20-30%,886,0.2371,0.4018,0.8909,0.8162,0.2909,0.4293,0.8943,0.8272,-0.0538,-0.0275,-0.0034,-0.0111
3,30-50%,1457,0.3438,0.5525,0.9179,0.8658,0.3933,0.5837,0.9231,0.8757,-0.0494,-0.0312,-0.0052,-0.0098
4,>50%,1182,0.5548,0.7257,0.9499,0.9179,0.6465,0.7814,0.9599,0.9355,-0.0917,-0.0558,-0.0101,-0.0176


In [ ]:
from bert_score import score as bert_score_fn
from rouge_score import rouge_scorer as rouge_scorer_lib
import sacrebleu

# ── per-protein ROUGE-L (fast, no model loading) ──────────────────────────────
rscorer = rouge_scorer_lib.RougeScorer(["rougeL"], use_stemmer=False)

def trunc(texts, tok):
    ids = tok(texts, padding="max_length", truncation=True,
               max_length=bert_truncate_length, return_tensors="pt")["input_ids"]
    return tok.batch_decode(ids, skip_special_tokens=True)

def per_protein_rougeL(preds, refs):
    return [rscorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]

# ── per-protein BLEU-4 (sentence-level) ───────────────────────────────────────
def per_protein_bleu4(preds, refs):
    return [
        sacrebleu.sentence_bleu(p, [r]).score / 100   # sacrebleu returns 0-100
        for p, r in zip(preds, refs)
    ]

# ── per-protein BERTScore (load model once per model type) ────────────────────
rt = RobertaTokenizer.from_pretrained(roberta_model)
bt = BertTokenizer.from_pretrained(biobert_model)

def per_protein_bertscore_rob(preds, refs):
    _, _, f1 = bert_score_fn(
        trunc(preds, rt), trunc(refs, rt),
        lang="en", verbose=True)
    return f1.tolist()

def per_protein_bertscore_bio(preds, refs):
    _, _, f1 = bert_score_fn(
        trunc(preds, bt), trunc(refs, bt),
        model_type=biobert_model, num_layers=biobert_num_layers, verbose=True)
    return f1.tolist()

# ── compute for each condition ─────────────────────────────────────────────────
# conditions: pred_with, pred_without, pred_baseline (baseline_predict)
# references always come from df["reference"]
conditions = {
    "with"     : df["pred_with"].tolist(),
    "without"  : df["pred_without"].tolist(),
}
# baseline only available for a subset of accessions
# align it to df.index, filling None where missing
conditions["predict_baseline"] = [
    baseline_predict.get(a) for a in df.index
]

refs = df["reference"].tolist()

for cond, preds in conditions.items():
    # for ext_baseline, only score where prediction exists
    valid_mask = [p is not None and isinstance(p, str) for p in preds]
    valid_accs  = [a for a, v in zip(df.index, valid_mask) if v]
    valid_preds = [p for p, v in zip(preds, valid_mask) if v]
    valid_refs  = [r for r, v in zip(refs, valid_mask) if v]

    print(f"\nscoring {cond} ({len(valid_accs):,} proteins)...")

    rl   = per_protein_rougeL(valid_preds, valid_refs)
    bl   = per_protein_bleu4(valid_preds, valid_refs)
    print(f"  computing RoBERTa BERTScore...")
    rb   = per_protein_bertscore_rob(valid_preds, valid_refs)
    print(f"  computing BioBERT BERTScore...")
    bio  = per_protein_bertscore_bio(valid_preds, valid_refs)

    # write into df, leaving NaN for proteins without a prediction
    df[f"rougeL_{cond}"]   = pd.Series(dict(zip(valid_accs, rl)),   dtype=float)
    df[f"bleu4_{cond}"]    = pd.Series(dict(zip(valid_accs, bl)),   dtype=float)
    df[f"bert_rob_{cond}"] = pd.Series(dict(zip(valid_accs, rb)),   dtype=float)
    df[f"bert_bio_{cond}"] = pd.Series(dict(zip(valid_accs, bio)),  dtype=float)

print("\nall per-protein scores added to df")

# ── save updated DataFrame ─────────────────────────────────────────────────────
export_path = os.path.join(out_dir, "all_predictions_analysis.csv")
df.to_csv(export_path)
print(f"saved {len(df):,} rows → {export_path}")
print(f"new score columns: {[c for c in df.columns if any(c.startswith(x) for x in ['rougeL_','bleu4_','bert_'])]}")


scoring with (4,203 proteins)...
  computing RoBERTa BERTScore...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/91 [00:00<?, ?it/s]

computing greedy matching.


...

done in 407.65 seconds, 10.31 sentences/sec
  computing BioBERT BERTScore...
calculating scores...
computing bert embedding.


  0%|          | 0/88 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/66 [00:00<?, ?it/s]

done in 373.05 seconds, 11.27 sentences/sec

all per-protein scores added to df
saved 4,203 rows → /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/foldseek_lora/drop01run/all_predictions_analysis.csv
new score columns: ['rougeL_with', 'bleu4_with', 'bert_rob_with', 'bert_bio_with', 'rougeL_without', 'bleu4_without', 'bert_rob_without', 'bert_bio_without', 'rougeL_predict_baseline', 'bleu4_predict_baseline', 'bert_rob_predict_baseline', 'bert_bio_predict_baseline']


In [ ]:
df.head()

,taxon,reference,best_hit_target,best_hit_fident,best_hit_qcov,best_hit_evalue,best_hit_desc,best_hit_org,pred_seq_only,pred_without,...,bert_rob_with,bert_bio_with,rougeL_without,bleu4_without,bert_rob_without,bert_bio_without,rougeL_predict_baseline,bleu4_predict_baseline,bert_rob_predict_baseline,bert_bio_predict_baseline
accession,,,,,,,,,,,,,,,,,,,,,
Q04W49,Leptospira,Necessary for normal cell division and for the maintenance of normal septation.,B0SC16,0.4690,0.8870,0.0000,Necessary for normal cell division and for the maintenance of normal septation.,Leptospira,Necessary for normal cell division and for the maintenance of normal septation.,Necessary for normal cell division and for the maintenance of normal septation.,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
Q8TR92,Methanosarcina,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Q8PUP4,0.9060,0.9850,0.0000,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Methanosarcina,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,Specifically methylates the uridine in position 2552 of 23S rRNA at the 2'-O position of the ribose in the fully ass...,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
O95905,Homo,Regulator of p53/TP53 stability and function. Inhibits MDM2-mediated degradation of p53/TP53 possibly by cooperating...,Q9W032,0.3050,0.9920,0.0000,Required in both the follicle cells and the germline for oocyte development.,Sophophora,May be required for efficient export of polyadenylated RNA.,May be required for efficient export of polyadenylated RNA.,...,0.8199,0.7615,0.0980,0.0000,0.8199,0.7615,0.0762,0.0000,0.8100,0.7216
Q70KD0,Micromonospora,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Q2MF16,0.5500,0.9520,0.0000,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Streptoalloteichus,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,Catalyzes the intramolecular carbocycle formation from D-glucose-6-phosphate to 2-deoxy-scyllo-inosose (DOI).,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
P48013,Schizosaccharomyces,"Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...",P26306,0.6480,0.9930,0.0000,"Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...",Schizosaccharomyces,"Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...","Acts in DNA repair and mutagenesis. Involved in promoting resistance to ionizing radiation and UV light, as well as ...",...,0.9707,0.9462,0.8197,0.6880,0.9707,0.9462,0.8197,0.6880,0.9707,0.9462
